# PyTorch/Lightning U-Net Implementation
Notebook to train a U-Net model on BraTS-style NIfTI data with early stopping model checkpoints, and detailed metric tracking.  Assumes the following directory structure:
```
dataset_root/
├── train/
│   ├── patient001/
│   │   ├── patient001_t1.nii.gz
│   │   ├── patient001_t1ce.nii.gz
│   │   ├── patient001_t2.nii.gz
│   │   ├── patient001_flair.nii.gz
│   │   └── patient001_seg.nii.gz
│   ├── patient002/ ...
│   └── ...
└── val/
    ├── patient050/ ...
    └── ...
```
This is not a *pure* PyTorch implementation of U-Net.  It uses ***MONAI*** for medical image I/O, pre-processing, and transforms.

---
<a name='setup_environment'></a>
## 1.0 <span style='color:blue'>|</span> Configure Environment

<a name='load_packages'></a>
### 1.1 <span style='color:blue'>|</span> Import Required Packages and Libraries

In [2]:
# Standard imports
import os, glob, random, warnings, gc
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'        # Fixes a warning from PyTorch
import torch
import numpy as np
import matplotlib.pyplot as plt

# Data pre-processing and Model definition
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.loggers import TensorBoardLogger
import nibabel as nib
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Spacingd,
    Orientationd,
    ScaleIntensityRanged,
    CropForegroundd,
    RandSpatialCropd,
    RandFlipd,
    RandRotate90d,
    ToTensord,
)
from monai.networks.nets import UNet
from monai.metrics import DiceMetric
from monai.losses import DiceLoss
from monai.inferers import sliding_window_inference
from monai.utils import set_determinism

# Make plots have guidelines
plt.style.use('ggplot')

# Squash Python warnings
warnings.filterwarnings('ignore')

# Enable Python's Garbage Collector
gc.collect()

2026-02-25 04:53:35.923916902 [W:onnxruntime:Default, device_discovery.cc:211 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:91 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"
<frozen importlib._bootstrap_external>:1325: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
2026-02-25 04:53:37.912586: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


556

<a name='global_variables'></a>
### 1.2 <span style='color:blue'>|</span> Declare Global Variables

In [3]:
# Directory Structure
ROOT_DIR  = '../BraTS/RSNA-ASNR-MICCAI-BraTS-2021'
TRAIN_DIR = os.path.join(ROOT_DIR, 'BraTS2021_TrainingSet/UPENN-GBM')
VAL_DIR   = os.path.join(ROOT_DIR, 'BraTS2021_ValidationSet/UPENN-GBM')

# Modalities and seg name
MODALITIES = ['t1', 't1ce', 't2', 'flair']
SEG_NAME = 'seg'

# Data Loading
BATCH_SIZE = 2
NUM_WORKERS = 4
IN_CHANNELS = len(MODALITIES)

OUT_CHANNELS = 4             # follows BraTS' 4-class definitions: 0 (background), 1 (WT), 2 (TC), 3 (ET)
DEVICE = 'gpu' if torch.cuda.is_available() else 'cpu'

# Random crop size for training
PATCH_SIZE = (128, 128, 128)

# Training
MAX_EPOCHS = 100
LEARNING_RATE = 1e-4
PATIENCE = 15                # Early stopping patience

# Define random seed value
SEED = 42

<a name='random_seed'></a>
### 1.3 <span style='color:blue'>|</span> Set Random Seed for Reproducibility

In [11]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
pl.seed_everything(SEED)
set_determinism(SEED)

# When running on CuDNN backend, it is recommended to set these two options
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.set_float32_matmul_precision('medium')

Seed set to 42


----
<a name='define_functions'></a>
## 2.0 <span style='color:blue'>|</span> Define Functions and Classes

<a name='dataset_class'></a>
### 2.1 <span style='color:blue'>|</span> Dataset Class (Lightning)
Loads the NIfTI files from each patient's directory.  If the path is the training path, then it collects the segmentation file.

In [6]:
class BraTSDataset(Dataset):
    def __init__(self, root_dir, modalities, seg_name, transform=None, is_train=True):
        self.root_dir = root_dir
        self.modalities = modalities
        self.seg_name = seg_name
        self.transform = transform
        self.is_train = is_train
        self.patients = sorted(glob.glob(os.path.join(root_dir, 'patient*')))

        # Collect all patient IDs
        self.patient_ids = [os.path.basename(p) for p in self.patients]

        # Filter out patients without all modalities or segmentation NIfTI volumes
        valid_patients = []
        for pid in self.patient_ids:
            patient_dir = os.path.join(root_dir, pid)
            files = glob.glob(os.path.join(patient_dir, f'{pid}_*.nii.gz'))
            required = [f'{pid}_{mod}.nii.gz' for mod in modalities]
            if not all(os.path.exists(os.path.join(patient_dir, f)) for f in required):
                print(f'Skipping {pid}: missing modalities')
                continue
            if is_train and not os.path.exists(os.path.join(patient_dir, f'{pid}_{seg_name}.nii.gz')):
                print(f'Skipping {pid}: missing segmentation')
                continue
            valid_patients.append(pid)

        self.patient_ids = valid_patients

    def __len__(self):
        return len(self.patient_ids)

    def __getitem__(self, idx):
        pid = self.patient_ids[idx]
        patient_dir = os.path.join(self.root_dir, pid)

        image_paths = [os.path.join(patient_dir, f'{pid}_{mod}.nii.gz') for mod in self.modalities]
        data = {mod: path for mod, path in zip(self.modalities, image_paths)}

        if self.is_train:
            seg_path = os.path.join(patient_dir, f'{pid}_{self.seg_name}.nii.gz')
            data['seg'] = seg_path

        # Apply transform (MONAI dict transforms)
        if self.transform:
            data = self.transform(data)

        # Return dict with keys: t1, t1ce, t2, flair, seg (optional)
        return data

<a name='data_module'></a>
### 2.2 <span style='color:blue'>|</span> Create a Lightning-style DataModule
This class provides methods to apply transforms to the datasets, the training and validation sets receive different transforms

In [8]:
class BraTSDataModule(pl.LightningDataModule):
    def __init__(
        self,
        train_dir,
        val_dir,
        modalities,
        seg_name,
        batch_size=1,
        num_workers=4,
        patch_size=(128, 128, 128),
    ):
        super().__init__()
        self.train_dir = train_dir
        self.val_dir = val_dir
        self.modalities = modalities
        self.seg_name = seg_name
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.patch_size = patch_size

        self.train_transform = self.get_train_transforms()
        self.val_transform = self.get_val_transforms()

    def get_train_transforms(self):
        return Compose(
            [
                LoadImaged(keys=['t1', 't1ce', 't2', 'flair', 'seg']),
                EnsureChannelFirstd(keys=['t1', 't1ce', 't2', 'flair', 'seg']),
                Spacingd(
                    keys=['t1', 't1ce', 't2', 'flair', 'seg'],
                    pixdim=(1.0, 1.0, 1.0),
                    mode=('bilinear', 'bilinear', 'bilinear', 'bilinear', 'nearest'),
                ),
                Orientationd(keys=['t1', 't1ce', 't2', 'flair', 'seg'], axcodes='RAS'),
                ScaleIntensityRanged(
                    keys=['t1', 't1ce', 't2', 'flair'],
                    a_min=0,
                    a_max=2000,
                    b_min=0.0,
                    b_max=1.0,
                    clip=True,
                ),
                # Crop foreground (recommended for limited GPU memory)
                CropForegroundd(
                    keys=['t1', 't1ce', 't2', 'flair', 'seg'],
                    source_key='t1',
                    allow_smaller=True,
                ),
                RandSpatialCropd(
                    keys=['t1', 't1ce', 't2', 'flair', 'seg'],
                    roi_size=self.patch_size,
                    random_size=False,
                ),
                RandFlipd(keys=['t1', 't1ce', 't2', 'flair', 'seg'], spatial_axis=0, prob=0.5),
                RandFlipd(keys=['t1', 't1ce', 't2', 'flair', 'seg'], spatial_axis=1, prob=0.5),
                RandRotate90d(keys=['t1', 't1ce', 't2', 'flair', 'seg'], prob=0.5),
                ToTensord(keys=['t1', 't1ce', 't2', 'flair', 'seg']),
            ]
        )

    def get_val_transforms(self):
        return Compose(
            [
                LoadImaged(keys=['t1', 't1ce', 't2', 'flair', 'seg']),
                EnsureChannelFirstd(keys=['t1', 't1ce', 't2', 'flair', 'seg']),
                Spacingd(
                    keys=['t1', 't1ce', 't2', 'flair', 'seg'],
                    pixdim=(1.0, 1.0, 1.0),
                    mode=('bilinear', 'bilinear', 'bilinear', 'bilinear', 'nearest'),
                ),
                Orientationd(keys=['t1', 't1ce', 't2', 'flair', 'seg'], axcodes='RAS'),
                ScaleIntensityRanged(
                    keys=['t1', 't1ce', 't2', 'flair'],
                    a_min=0,
                    a_max=2000,
                    b_min=0.0,
                    b_max=1.0,
                    clip=True,
                ),

                CropForegroundd(
                    keys=['t1', 't1ce', 't2', 'flair', 'seg'],
                    source_key='t1',
                    allow_smaller=True,
                ),
                ToTensord(keys=['t1', 't1ce', 't2', 'flair', 'seg']),
            ]
        )

    def setup(self, stage=None):
        # Called by trainer.fit()
        if stage in (None, 'fit'):
            self.train_ds = BraTSDataset(
                self.train_dir, self.modalities, self.seg_name, transform=self.train_transform, 
                is_train=True
            )
            self.val_ds = BraTSDataset(
                self.val_dir, self.modalities, self.seg_name, transform=self.val_transform, 
                is_train=True
            )
        if stage == 'validate':
            self.val_ds = BraTSDataset(
                self.val_dir, self.modalities, self.seg_name, transform=self.val_transform, 
                is_train=True
            )

    def train_dataloader(self):
        return DataLoader(
            self.train_ds,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
            pin_memory=True if DEVICE == 'gpu' else False,
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_ds,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=True if DEVICE == 'gpu' else False,
        )

<a name='UNet_module'></a>
### 2.3 <span style='color:blue'>|</span> Define the UNet/Lightning Module

In [17]:
class UNetLightningModule(pl.LightningModule):
    def __init__(
        self,
        in_channels,
        out_channels,
        learning_rate=1e-4,
    ):
        super().__init__()
        self.save_hyperparameters()

        self.model = UNet(
            spatial_dims=3,
            in_channels=in_channels,
            out_channels=out_channels,
            channels=(16, 32, 64, 128, 256),
            strides=(2, 2, 2, 2),
            num_res_units=2,
            dropout=0.2,
        )

        self.loss_function = DiceLoss(to_onehot_y=True, softmax=True)
        self.dice_metric = DiceMetric(reduction='mean_batch')  # per class dice
        self.learning_rate = learning_rate

        # Store metrics for epoch-level logging
        self.train_losses = []
        self.val_losses = []
        self.val_dice_per_epoch = []

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        images = torch.cat([batch[mod] for mod in MODALITIES], dim=1)  # [B, 4, H, W, D]
        labels = batch['seg']
        outputs = self(images)

        loss = self.loss_function(outputs, labels)
        self.log('train_loss', loss, prog_bar=True, on_step=True, on_epoch=True)
        self.train_losses.append(loss.item())
        return loss

    def validation_step(self, batch, batch_idx):
        images = torch.cat([batch[mod] for mod in MODALITIES], dim=1)
        labels = batch['seg']
        outputs = self(images)

        loss = self.loss_function(outputs, labels)
        self.val_losses.append(loss.item())

        # Compute Dice per class
        pred_argmax = torch.argmax(outputs, dim=1, keepdim=True)
        self.dice_metric(y_pred=pred_argmax, y=labels)
        dice_scores = self.dice_metric.compute()  # shape: [num_classes]
        self.dice_metric.reset()
        # dice_scores: tensor([dice_bg, diceWT, diceTC, diceET])

        self.log('val_loss', loss, prog_bar=True)
        # Log per-class dice
        for i, name in enumerate(['dice_bg', 'dice_WT', 'dice_TC', 'dice_ET']):
            self.log(f'val_{name}', dice_scores[i], prog_bar=False)

        # return per-batch dice average (mean over labels, excluding background)
        dice_mean = dice_scores[1:].mean()  # exclude background (index 0)
        self.log('val_dice', dice_mean, prog_bar=True)
        return loss

    def on_train_epoch_end(self):
        avg_loss = np.mean(self.train_losses)
        self.log('train_loss_epoch', avg_loss)
        self.train_losses.clear()  # free memory

    def on_validation_epoch_end(self):
        avg_loss = np.mean(self.val_losses)
        self.log('val_loss_epoch', avg_loss)
        self.val_losses.clear()

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.2, patience=5)
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'monitor': 'val_loss_epoch',
            },
        }

<a name='training_function'></a>
### 2.4 <span style='color:blue'>|</span> Training Function

In [15]:
def train():

    # Data
    datamodule = BraTSDataModule(
        train_dir=TRAIN_DIR,
        val_dir=VAL_DIR,
        modalities=MODALITIES,
        seg_name=SEG_NAME,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        patch_size=PATCH_SIZE,
    )

    # Model
    model = UNetLightningModule(
        in_channels=IN_CHANNELS,
        out_channels=OUT_CHANNELS,
        learning_rate=LEARNING_RATE,
    )

    # Callbacks
    checkpoint_callback = ModelCheckpoint(
        dirpath='checkpoints/',
        filename='unet-brats-{val_dice:.4f}',
        save_top_k=3,
        monitor='val_dice',
        mode='max',
        save_weights_only=True,
    )

    early_stop_callback = EarlyStopping(
        monitor='val_dice',
        patience=PATIENCE,
        mode='max',
        verbose=True,
    )

    # Logger (TensorBoard)
    logger = TensorBoardLogger('tb_logs', name='unet_brats')

    # Trainer
    trainer = pl.Trainer(
        max_epochs=MAX_EPOCHS,
 #       gpu=1 if torch.cuda.is_available() else 0,
        callbacks=[checkpoint_callback, early_stop_callback],
        logger=logger,
        log_every_n_steps=5,
        enable_checkpointing=True,
        enable_progress_bar=True,
        precision=16 if torch.cuda.is_available() else 32,  # mixed precision for speed
    )

    # Train!
    trainer.fit(model, datamodule)

    best_model_path = checkpoint_callback.best_model_path
    if best_model_path:
        print(f'Best model saved at: {best_model_path}')

        best_model = UNetLightningModule.load_from_checkpoint(best_model_path)
        trainer.validate(model=best_model, datamodule=datamodule)

    print('Training completed.')

In [18]:
train()

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name          ┃ Type     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model         │ UNet     │  4.8 M │ train │     0 │
│ 1 │ loss_function │ DiceLoss │      0 │ train │     0 │
└───┴───────────────┴──────────┴────────┴───────┴───────┘

Trainable params: 4.8 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 4.8 M                                                                                                
Total estimated model params size (MB): 19                                                                         
Modules in train mode: 145                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

ValueError: num_samples should be a positive integer value, but got num_samples=0